# Lazy-кандген: обучение и результаты (Google Colab)

Устройство модели — `docs/lazy.md`: энкодер истории + Qwen по запросу с cross-attention. Ячейки запускать
**сверху вниз**. Всё, что считается, пишется на Google Drive в папку прогона `MyDrive/recsys/artifacts/lazy/<RUN>/`
и переживает отключение Colab.

**Перед запуском:** Runtime → Change runtime type → **A100 GPU**. На Drive в `MyDrive/recsys/` должны лежать
`music4all_crs/` (7 parquet-файлов) и `artifacts/semantic_ids/` (`semantic_codes.npy`, `rqvae_checkpoint.pt`).

| шаг | ячейки | время на A100 |
|---|---|---|
| 0. подготовка | 1–4 | ~5 мин, **в каждой новой сессии** |
| 0'. векторы названий треков | 4a | ~3 мин, **один раз** на все прогоны |
| 1. обучение | 5 | 2 эпохи; время — в логе (`с/шаг`, «до конца этапа»); при обрыве — снова 1–4 и 5 |
| 2. результаты | 6–10 | ~1 ч |
| 3. выключить машину | 11 | — |

Прогоны эксперимента — по одному на сессию, каждый в своей папке (ячейка 4): `e0` (без энкодера), `e2` (2 слоя
энкодера), `prompt` (decoder-only бейзлайн). Можно запустить ячейки 5–11 сразу одну за другой (Shift+Enter):
они встанут в очередь, в конце машина выключится. Вкладку не закрывать, компьютеру не засыпать.


## 0. Подготовка (в каждой новой сессии)

In [ ]:
# 1. Drive, код (ветка dev/lazy), датасет на диск машины, artifacts — ссылкой на Drive
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/rimgro/conversational-recsys.git"
BRANCH = "dev/lazy"
DRIVE_DIR = "/content/drive/MyDrive/recsys"      # данные и результаты на Google Drive
REPO_DIR = "/content/recsys-lazy"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    url = REPO_URL
    try:                                         # приватный репозиторий: секрет GITHUB_TOKEN (ключик слева)
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, url, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
    need = [f"music4all_crs/{n}.parquet" for n in ("tracks_meta", "train", "train_queries", "train_qrels",
            "test_public", "test_public_queries", "test_public_qrels")]
    need += ["artifacts/semantic_ids/semantic_codes.npy", "artifacts/semantic_ids/rqvae_checkpoint.pt"]
    missing = [p for p in need if not os.path.exists(f"{DRIVE_DIR}/{p}")]
    if missing:
        raise FileNotFoundError("на Google Drive нет файлов (загрузите их в эти папки и запустите ячейку снова):\n  "
                                + "\n  ".join(f"{DRIVE_DIR}/{p}" for p in missing))
    # датасет — на локальный диск VM (с Drive чтение медленное); artifacts — ссылкой на Drive (результаты сохранятся)
    subprocess.run(["rsync", "-a", f"{DRIVE_DIR}/music4all_crs/", "music4all_crs/"], check=True)
    os.makedirs(f"{DRIVE_DIR}/artifacts/lazy", exist_ok=True)
    if not os.path.islink("artifacts"):
        os.symlink(f"{DRIVE_DIR}/artifacts", "artifacts")
print("папка:", os.getcwd(), "| ветка:", subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
      capture_output=True, text=True).stdout.strip())

In [ ]:
# 2. пакеты: пайплайн (requirements.txt) и обучение; torchao из Colab старый — с ним peft падает при создании LoRA
%pip install -q -r requirements.txt "transformers>=4.51" "peft>=0.13"
%pip uninstall -y -q torchao

In [ ]:
# 3. файлы и GPU
import torch
for p in ["music4all_crs/train.parquet", "music4all_crs/train_queries.parquet", "music4all_crs/tracks_meta.parquet",
          "music4all_crs/test_public.parquet", "artifacts/semantic_ids/semantic_codes.npy",
          "artifacts/semantic_ids/rqvae_checkpoint.pt"]:
    print(("ok  " if os.path.exists(p) else "НЕТ ") + p)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет — Runtime -> Change runtime type")

In [ ]:
# 4. прогон: раскомментируйте ОДИН. У каждого своя папка: в папку с другой моделью обучение не запустится
RUN, EXTRA = "artifacts/lazy/e0", "--config configs/lazy.yaml"                                          # без энкодера
# RUN, EXTRA = "artifacts/lazy/e2", "--config configs/lazy.yaml --set twotower.lazy.encoder_layers=2"   # 2 слоя энкодера
# RUN, EXTRA = "artifacts/lazy/prompt", "--config configs/lazy.yaml --set twotower.lazy.mode=prompt"     # decoder-only
os.makedirs(RUN, exist_ok=True)
print("прогон:", RUN, "|", EXTRA, "| уже есть:", sorted(os.listdir(RUN)))


In [ ]:
# 4a. векторы названий треков «артист — название» (Qwen3-Embedding) — один раз, общие для всех прогонов
if not os.path.exists("artifacts/item_text/qwen3emb.npz"):
    !python scripts/encode_item_text.py
print("векторы названий:", os.path.exists("artifacts/item_text/qwen3emb.npz"))


## 1. Обучение
`--resume`: если в папке уже есть `checkpoint.pt` (сессия оборвалась), продолжает с него; если обучение закончено —
не обучает заново, только пересчитывает `metrics.json`. Раз в 500 шагов — валидация и `checkpoint.pt` на Drive.


In [ ]:
# 5. обучение: этап A (0.3 эпохи, Qwen заморожена) -> этап B (1.7 эпохи, + LoRA). В начале — FLOPs на запрос;
# в логе: loss, τ, гейт (насколько декодер уже смотрит на историю), с/шаг, сколько осталось до конца этапа
!python scripts/train_twotower.py --resume --run-dir {RUN} {EXTRA}


## 2. Результаты (всё пишется в папку прогона на Drive)

In [ ]:
# 6. валидация: лучшая и по типам запросов -> metrics.json
import json
m = json.load(open(f"{RUN}/metrics.json"))
print("лучшее на валидации:", m["best_val"])
{t: {k: round(v, 3) for k, v in d.items()} for t, d in m["last_val"]["by_type"].items()}

In [ ]:
# 7. векторы треков и запросов частей ranker_train, valid, test, submit (twotower.encode.parts; ranker_train и valid —
# из train: на них учится ранкер, без них кандген в пайплайне пуст) + recall по каталогу -> recall_<часть>.json (~40 мин)
!python scripts/encode_twotower.py --run-dir {RUN}

In [ ]:
# 8. разбор: различимы ли треки, покрытие выдачи, recall для целей из истории и новых -> analyze.txt
!python scripts/analyze_twotower.py --run-dir {RUN} | tee {RUN}/analyze.txt

In [ ]:
# 9. кандген в пайплайне (только twotower) на частях valid и test -> candgen_eval/<часть>/report.md
!python -m recsys candgen --sources twotower --set candgen.sources.twotower.dir={RUN} --set run.dir={RUN} --run candgen_eval
# примеры выдачи -> examples.txt
!python scripts/show_twotower.py --run-dir {RUN} -n 10 | tee {RUN}/examples.txt
!python scripts/show_twotower.py --run-dir {RUN} --type similar_to -n 3 | tee -a {RUN}/examples.txt

In [ ]:
# 10. (по желанию, ~25 мин на длину) сколько даёт история: та же модель с историей 0 и 16 треков -> h0/, h16/
for L in (0, 16):
    !python scripts/encode_twotower.py --run-dir {RUN} --history-len {L} --out {RUN}/h{L}

## 3. Выключить машину (вычислительные единицы больше не тратятся; Drive сохраняется)

In [ ]:
# 11.
from google.colab import drive, runtime
drive.flush_and_unmount()   # дописать файлы на Drive
runtime.unassign()          # отключить и удалить машину

## Что лежит в папке прогона после всего

| файл | что |
|---|---|
| `model.pt`, `last.pt`, `checkpoint.pt`, `config.yaml` | лучшая модель, последняя, для продолжения (с оптимизатором), конфиг |
| `log.jsonl`, `metrics.json` | лог обучения; валидация (лучшая и по типам), `flops` — FLOPs на запрос |
| `item_vectors.npy`, `item_ids.npy`, `query_vectors_<часть>.npz` | векторы — по ним работает пайплайн без GPU |
| `recall_<часть>.json` | recall@10/100/200 по всему каталогу на части, по типам |
| `analyze.txt`, `examples.txt`, `candgen_eval/` | разбор, примеры, отчёт кандгена в пайплайне |
| `h0/`, `h16/` | recall при истории 0 и 16 треков |

## Другие прогоны
Новая сессия: ячейки 1–3, в ячейке 4 — другая строка (или своё имя и параметры), затем 5–11. Ещё ablation:
- реже cross-attention: `RUN, EXTRA = "artifacts/lazy/e0_x4", "--config configs/lazy.yaml --set twotower.lazy.cross_every=4"`
- без названия трека: `RUN, EXTRA = "artifacts/lazy/e0_notext", "--config configs/lazy.yaml --set twotower.item.text.enabled=false"`

## Дальше — в пайплайне (CPU)
```
python -m recsys tune-policy --sources +twotower --set candgen.sources.twotower.dir=<RUN> --run lazy
python -m recsys run --sources +twotower --set candgen.sources.twotower.dir=<RUN> --config runs/lazy/selection.yaml --run lazy
```
Скоры в признаки CatBoost не идут (`features: false`): модель учится на всём train.
